In [ ]:
from pathlib import Path
import sys

# --- Resolve repo root (works whether you run from repo root or within notebooks/) ---
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    if (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    elif (REPO_ROOT.parent.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent.parent

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

from IPython.display import display
from notebooks.set_up import (
    ensure_dirs,
    SYNTH_DIR,
    TABLE_DIR, FIG_DIR,
    CATE_DIR, 
)

ensure_dirs()

# Inputs
base_path = str(SYNTH_DIR) + "/"
# Outputs
output_result = str(CATE_DIR) + "/"
#output_figure = str(FIG_DIR) + "/"

Estimators and evaluation utilities live in `src/causalmix/cate/`.

In [ ]:
# CATE estimators + evaluation
from causalmix.cate import *

# Application 2. Hyperparameter selection with synthetic mcrpc datasets

In [0]:
# run many causal-forest hyperparameter configurations (min.node.size × num.trees)
# Replaces previous mixed-estimator list with a grid of causal forest variants for Application 2.
#
# Assumptions (unchanged):
# - preprocess_data, causal_forest, evaluate_estimator_rep, summarize_results exist in the environment
# - causal_forest(...) accepts keyword args num_trees and min_node_size and returns a result dict
#   compatible with evaluate_estimator_rep (including OOB CATE predictions).
# - Spark session `spark` is available.

import numpy as np
import pandas as pd
import time
from pyspark.sql import functions as F

# -----------------------
# User paths / settings
# -----------------------
n_reps = 50
target_concurrency = 32

OUTCOME = "hosp_ed_any"
TREATMENT = "exp"

# -----------------------
# Causal forest grid (only CF variants)
# -----------------------
MIN_NODE_SIZES = (5, 10, 20, 40, 80)
NUM_TREES = (500, 1000, 2000, 4000)

# Build estimator configs: (name, encoding, type, params)
ESTIMATOR_CONFIGS = []
for mn in MIN_NODE_SIZES:
    for nt in NUM_TREES:
        name = f"CausalForest_min{mn}_trees{nt}"
        # choose encoding that matches your usual pipeline; onehot is fine for grf-like forests
        ESTIMATOR_CONFIGS.append((name, "onehot", "cf", {"min_node_size": mn, "num_trees": nt}))

# -----------------------
# 1) Build one Spark DF with stable idx (driver-side)
# -----------------------
data_parts = []
truth_parts = []

for rep in range(n_reps):
    # Load delta -> pandas (order preserved within pandas)
    pdf = spark.read.format("delta").load(base_path + f"df_bias0_gen_{rep}").toPandas()
    pdf = pdf.reset_index(drop=True)
    pdf["idx"] = np.arange(len(pdf), dtype=int)
    pdf["rep"] = rep
    data_parts.append(pdf)

    # Load truth arrays (must match pandas row order)
    true_ate = np.load("/dbfs" + base_path + f"ate_bias0_{rep}.npy")
    true_ate = float(true_ate.item() if hasattr(true_ate, "item") else true_ate)

    true_cate = np.load("/dbfs" + base_path + f"ite_bias0_{rep}.npy")
    true_cate = np.asarray(true_cate, dtype=float)

    truth_pdf = pd.DataFrame({
        "rep": rep,
        "idx": np.arange(len(true_cate), dtype=int),
        "true_cate": true_cate,
        "true_ate": true_ate,
    })
    truth_parts.append(truth_pdf)

data_all_pdf = pd.concat(data_parts, ignore_index=True)
truth_all_pdf = pd.concat(truth_parts, ignore_index=True)

data_sdf = spark.createDataFrame(data_all_pdf)
truth_sdf = spark.createDataFrame(truth_all_pdf)

# Join so each row carries truth
joined = data_sdf.join(truth_sdf, on=["rep", "idx"], how="inner")

# -----------------------
# 2) Pandas UDF per rep (NO spark usage inside)
# -----------------------
def eval_one_rep(pdf: pd.DataFrame) -> pd.DataFrame:
    rep = int(pdf["rep"].iloc[0])

    true_ate = float(pdf["true_ate"].iloc[0])
    true_cate = pdf.sort_values("idx")["true_cate"].to_numpy(dtype=float)

    # drop truth columns, keep outcome/treatment/covariates
    df = pdf.sort_values("idx").drop(columns=["true_ate", "true_cate"])

    # preprocess once per rep (onehot/dummy as used below)
    df_onehot = preprocess_data(df, dummy_code=False)
    df_dummy  = preprocess_data(df, dummy_code=True)

    rows = []
    for est_name, encoding, est_type, params in ESTIMATOR_CONFIGS:
        try:
            df_est = df_onehot if encoding == "onehot" else df_dummy
            df_est = df_est.copy(deep=True)

            t0 = time.perf_counter()
            if est_type == "cf":
                # call your causal_forest wrapper (must accept min_node_size & num_trees)
                res = causal_forest(
                    df_est,
                    outcome=OUTCOME,
                    treatment=TREATMENT,
                    covariates=None,
                    alpha=0.05,
                    num_trees=params["num_trees"],
                    min_node_size=params["min_node_size"],
                )
            else:
                raise ValueError(f"Estimator type '{est_type}' not supported in this HPO script.")

            res["runtime_sec"] = time.perf_counter() - t0

            # evaluate_estimator_rep accept res and return a dict/Series with required metrics
            m = evaluate_estimator_rep(res, true_cate=true_cate, true_ate=true_ate, estimator_name=est_name)
            m["rep"] = rep
            m["n"] = int(len(df_est))
            m["error"] = ""
            rows.append(m)

        except Exception as e:
            rows.append({
                "estimator": est_name,
                "runtime_sec": np.nan,
                "ate_hat": np.nan,
                "ate_bias": np.nan,
                "ate_stderr": np.nan,
                "ate_coverage": np.nan,
                "cate_rmse": np.nan,
                "cate_bias_abs_mean": np.nan, # update 2/23/2026: add mean absolute bias
                "cate_ci_width_mean": np.nan, # update 2/23/2026: add mean CI width
                "cate_coverage": np.nan,
                "rep": rep,
                "n": int(len(df)),
                "error": str(e),
            })

    return pd.DataFrame(rows)

schema = """
estimator string,
runtime_sec double,
ate_hat double,
ate_bias double,
ate_stderr double,
ate_coverage int,
cate_rmse double,
cate_bias_abs_mean double,
cate_ci_width_mean double,
cate_coverage double,
rep int,
n int,
error string
"""

# -----------------------
# 3) Run distributed evaluation
# -----------------------
per_rep_spark = (joined
    .repartition(target_concurrency, "rep")
    .groupBy("rep")
    .applyInPandas(eval_one_rep, schema=schema)
)

per_rep_df = per_rep_spark.toPandas()
# drop rows where metrics are missing (failed fits)
per_rep_df_valid = per_rep_df.dropna(subset=["ate_bias", "cate_rmse"])

summary_df = summarize_results(per_rep_df_valid)

per_rep_df.round(3).to_csv(output_result + "per_rep_hp_results.csv", index=False)
summary_df.round(3).to_csv(output_result + "summary_hp_results.csv", index=False)

print(summary_df)  # replaced display()

estimator,n_reps,runtime_mean_sec,runtime_sd_sec,ate_rmse,ate_bias_mean,ate_bias_sd,ate_stderr_mean,ate_coverage,cate_rmse_mean,cate_rmse_sd,cate_bias_abs_mean_mean,cate_bias_abs_mean_sd,cate_ci_width_mean_mean,cate_ci_width_mean_sd,cate_coverage_mean
CausalForest_min10_trees1000,50,11.859408557440002,0.37638443855916714,0.00889821907061982,9.22119125897787E-5,0.008988075978076299,0.00884366895684915,0.92,0.03061658636906163,0.004121056856102961,0.023983890249287727,0.0032703755244500128,0.09564459977284835,0.004231352484312339,0.8515714982918497
CausalForest_min10_trees2000,50,22.878830731040033,0.6709015946221719,0.00895631580268414,3.1919653624405545E-4,0.009041497659525414,0.008816573654886029,0.94,0.03060136294263291,0.004048532213392102,0.023991335686086735,0.0032197985400630575,0.09219117815476457,0.004491409583862949,0.8425475841874085
CausalForest_min10_trees4000,50,46.5439467239,1.657338449466587,0.008918411142651256,3.9616687367246607E-4,0.009000062874926107,0.008804779201784591,0.94,0.03059303276268551,0.004081434356971593,0.023980334219788967,0.0032451976323635196,0.09127686796745788,0.004510806225989327,0.8441825280624695
CausalForest_min10_trees500,50,7.212060699780014,0.8456138649064193,0.008869950472396419,-9.279965129809872E-5,0.008959512651166505,0.008899861782758596,0.94,0.03078706952070625,0.004081843488194505,0.024121713706415596,0.0032170686277290853,0.10179097183026763,0.0035580792294209814,0.868818936066374
CausalForest_min20_trees1000,50,11.215167496999992,0.4955939375614315,0.008911605704545796,6.832413602378796E-5,0.009001816599302576,0.008844478999618958,0.92,0.02693027784448463,0.0037859948034804354,0.021211495099164816,0.0031651731684787894,0.07653522828462898,0.0049218076317118845,0.8179892630551489
CausalForest_min20_trees2000,50,21.11551539656005,0.7817512144233086,0.00896630050781177,2.967620383774096E-4,0.009052369015051078,0.008817445457326713,0.94,0.026951086378717064,0.003741010602198992,0.02125868658073426,0.0031279283872135757,0.07533367274354466,0.004944123541313295,0.8168130795510006
CausalForest_min20_trees4000,50,42.68228247594,1.6415357017875238,0.008926658957357952,3.725390563393299E-4,0.009009431271121502,0.00880585589649979,0.94,0.026935046017927365,0.003732742569382262,0.021246244657688117,0.0031290934541119515,0.07543704011072189,0.005005036117109249,0.8232601268911663
CausalForest_min20_trees500,50,6.938617374560026,1.0807405732599773,0.008885711784179098,-1.2050460412749376E-4,0.008975098916621828,0.008900558098118827,0.94,0.027077513272984036,0.0037845345702842135,0.021319899081483837,0.0031487166429149767,0.07964321403407353,0.004461429158759604,0.8296339677891654
CausalForest_min40_trees1000,50,10.393731433899994,0.6531356306874408,0.008924688272910316,3.531771739501699E-5,0.00901522597696677,0.008845242553608324,0.92,0.024615223674858466,0.0033881085577213347,0.019390909403339575,0.002980816826600775,0.06376478921110207,0.005851269679022091,0.7888091752074183
CausalForest_min40_trees2000,50,18.990314961379973,0.7066925721883975,0.008977810983705993,2.636480586425936E-4,0.009065047235844262,0.008818308301684218,0.94,0.02461977683616533,0.0033635877256824164,0.01941651354183443,0.002970601368781578,0.06353619485116092,0.005887627234876893,0.7924987798926305
